## 연습 1: 카탈로그 구조 설정

여러분은 학사 운영을 디지털화하고 있는 중간 규모 기관인 **Lakeside University**의 데이터 엔지니어입니다. 여러분의 팀은 학생 레코드, 강좌 카탈로그, 수강 신청 데이터를 관리하기 위해 Azure Databricks와 Unity Catalog에 데이터 플랫폼을 구축하고 있습니다.

이 연습에서는 **메달리온 아키텍처** 명명 패턴을 따르는 하나의 `catalog`와 세 개의 `schema`로 구성된 최상위 네임스페이스 구조를 만듭니다.

다음은 여러분이 만들게 될 전체 Unity Catalog 구조입니다:

```txt

Unity Catalog
└── edu_dev  [catalog]
    │   tags: environment=development | university=lakeside | data_classification=internal
    │
    ├── bronze  [schema]
    │   └── raw_files/  [managed volume]
    │       └── students.csv
    │
    ├── silver  [schema]
    │   ├── students  [table]
    │   │   ├── student_id       BIGINT  NOT NULL  ← PK
    │   │   ├── first_name       STRING
    │   │   ├── last_name        STRING
    │   │   ├── email            STRING
    │   │   ├── enrollment_year  INT
    │   │   ├── program          STRING
    │   │   └── phone_number     STRING  (added in Exercise 6)
    │   │
    │   ├── courses  [table]
    │   │   ├── course_id    BIGINT  NOT NULL  ← PK
    │   │   ├── course_name  STRING
    │   │   ├── department   STRING
    │   │   └── credits      INT
    │   │
    │   ├── enrollments  [table]
    │   │   ├── enrollment_id  BIGINT  NOT NULL  ← PK
    │   │   ├── student_id     BIGINT            → FK → students.student_id
    │   │   ├── course_id      BIGINT            → FK → courses.course_id
    │   │   ├── semester       STRING
    │   │   └── grade          DECIMAL(4,2)
    │   │
    │   ├── vw_student_enrollments  [view]
    │   │   └── joins students + courses + enrollments
    │   │
    │   └── get_grade_classification(grade)  [function]
    │       └── returns STRING  (A / B / C / D / F)
    │
    └── gold  [schema]
        └── vw_department_enrollment_stats  [materialized view]
            └── aggregates enrollments + courses
                (department, total_enrollments, avg_grade, distinct_students)
```

### 작업 1.1 — 개발 카탈로그 생성

설명적인 주석과 함께 `edu_dev`라는 카탈로그를 만드세요. 이 카탈로그는 Lakeside University 데이터 플랫폼의 **개발 환경** 역할을 합니다.

**명명 규칙 참고 사항:**
- 소문자와 밑줄을 사용하세요
- 마침표, 공백, 슬래시 또는 제어 문자를 사용하지 마세요

> 🤖 **Genie Code 팁:** Genie Code에 물어보세요:
> *"주석과 함께 Databricks SQL에서 Unity Catalog 카탈로그를 만들려면 어떻게 해야 해?"*

**힌트:** `CREATE CATALOG IF NOT EXISTS` 다음에 `COMMENT` 문자열을 사용하세요.

In [ ]:
# 기본 저장소가 활성화되어 있지 않으므로 기본 카탈로그의 루트에서 저장소 경로를 상속합니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"저장소 루트: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS edu_dev
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Lakeside University 데이터 플랫폼의 개발 환경'
""")

### 작업 1.2 — 메달리온 스키마 생성

`edu_dev` 카탈로그 내에 **메달리온 아키텍처**를 구현하는 세 개의 스키마를 만드세요:

| 스키마 | 목적 |
|--------|---------|
| `bronze` | 수집된 원시 데이터 — 수정되지 않은 소스 파일 |
| `silver` | 정제, 검증, 보강된 데이터 |
| `gold` | 집계되어 분석 준비가 완료된 데이터 |

> 🤖 **Genie Code 팁:** 물어보세요:
> *"SQL을 사용하여 Databricks Unity Catalog 카탈로그에 여러 스키마를 만드는 방법을 보여줘."*

**힌트:** 각 스키마에 대해 `CREATE SCHEMA IF NOT EXISTS catalog.schema_name`을 사용하세요.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS edu_dev.bronze
COMMENT '수집된 원시 데이터 — 수정되지 않은 소스 파일';

CREATE SCHEMA IF NOT EXISTS edu_dev.silver
COMMENT '정제, 검증, 보강된 데이터';

CREATE SCHEMA IF NOT EXISTS edu_dev.gold
COMMENT '집계되어 분석 준비가 완료된 데이터';

## 연습 2: 제약 조건이 있는 테이블 생성

카탈로그 구조가 준비되었으니 `silver` 스키마에 Lakeside University의 학사 데이터를 위한 핵심 테이블을 만드세요. 데이터 모델을 문서화하기 위해 **기본 키**와 **외래 키** 제약 조건을 정의합니다. Unity Catalog에서 이러한 제약 조건은 정보 제공용이라는 점에 유의하세요. 즉, 쿼리 최적화에는 도움이 되지만 쓰기 시점에 참조 무결성을 강제하지는 않습니다.

### 작업 2.1 — `students` 테이블 생성

다음 열을 사용하여 학생 레코드를 저장하는 관리형 Delta 테이블 `edu_dev.silver.students`를 만드세요:

| 열 | 유형 | 설명 |
|--------|------|-------------|
| `student_id` | BIGINT | 고유 학생 식별자(기본 키) |
| `first_name` | STRING | 학생의 이름 |
| `last_name` | STRING | 학생의 성 |
| `email` | STRING | 대학 이메일 주소 |
| `enrollment_year` | INT | 학생이 입학한 연도 |
| `program` | STRING | 학위 프로그램 |

> 🤖 **Genie Code 팁:** 물어보세요:
> *"Unity Catalog에서 기본 키 제약 조건이 있는 관리형 Delta 테이블을 만들려면 어떻게 해야 해?"*

**힌트:** 열 목록 끝에 `CONSTRAINT students_pk PRIMARY KEY (student_id)`를 추가하세요.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS edu_dev.silver.students (
  student_id      BIGINT NOT NULL,
  first_name      STRING,
  last_name       STRING,
  email           STRING,
  enrollment_year INT,
  program         STRING,
  CONSTRAINT students_pk PRIMARY KEY (student_id)
);

### 작업 2.2 — `courses` 테이블 생성

다음 열로 `edu_dev.silver.courses`를 만드세요:

| 열 | 유형 | 설명 |
|--------|------|-------------|
| `course_id` | BIGINT | 고유 강좌 식별자(기본 키) |
| `course_name` | STRING | 강좌의 전체 이름 |
| `department` | STRING | 강좌를 제공하는 학과 |
| `credits` | INT | 학점 수 |

> 🤖 **Genie Code 팁:** 물어보세요:
> *"기본 키 제약 조건이 있는 Databricks Unity Catalog의 courses 테이블에 대한 CREATE TABLE 문을 작성해 줘."*

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS edu_dev.silver.courses (
  course_id   BIGINT NOT NULL,
  course_name STRING,
  department  STRING,
  credits     INT,
  CONSTRAINT courses_pk PRIMARY KEY (course_id)
);

### 작업 2.3 — 외래 키가 있는 `enrollments` 테이블 생성

어떤 학생이 어떤 강좌에 등록되어 있는지 기록하는 `edu_dev.silver.enrollments`를 만드세요. 이 테이블은 외래 키 제약 조건을 통해 `students`와 `courses`를 모두 참조해야 합니다.

| 열 | 유형 | 설명 |
|--------|------|-------------|
| `enrollment_id` | BIGINT | 고유 수강 신청 레코드(기본 키) |
| `student_id` | BIGINT | `students.student_id`를 참조 |
| `course_id` | BIGINT | `courses.course_id`를 참조 |
| `semester` | STRING | 학기(예: `Spring 2023`) |
| `grade` | DECIMAL(4,2) | 0.0~10.0 척도의 숫자 성적 |

> 🤖 **Genie Code 팁:** 물어보세요:
> *"Databricks Unity Catalog에서 테이블을 만들 때 다른 테이블을 참조하는 외래 키 제약 조건을 추가하려면 어떻게 해야 해?"*

**힌트:** 각 외래 키에 대해 `CONSTRAINT <name> FOREIGN KEY (<col>) REFERENCES <catalog>.<schema>.<table>(<col>)`를 사용하세요.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS edu_dev.silver.enrollments (
  enrollment_id BIGINT NOT NULL,
  student_id    BIGINT,
  course_id     BIGINT,
  semester      STRING,
  grade         DECIMAL(4,2),
  CONSTRAINT enrollments_pk         PRIMARY KEY (enrollment_id),
  CONSTRAINT enrollments_students_fk FOREIGN KEY (student_id) REFERENCES edu_dev.silver.students(student_id),
  CONSTRAINT enrollments_courses_fk  FOREIGN KEY (course_id)  REFERENCES edu_dev.silver.courses(course_id)
);

### 샘플 데이터 — 이 셀을 실행하세요

다음 셀은 세 개의 테이블에 샘플 데이터를 삽입합니다. 연습 3으로 계속하기 전에 **수정 없이 실행하세요**.

In [ ]:
%sql
-- 학생 데이터 채우기
INSERT INTO edu_dev.silver.students VALUES
(1001, 'Emma',   'Watson',    'emma.watson@lakeside.edu',    2022, 'Computer Science'),
(1002, 'Liam',   'Ahmed',     'liam.ahmed@lakeside.edu',     2023, 'Data Science'),
(1003, 'Sofia',  'Chen',      'sofia.chen@lakeside.edu',     2021, 'Mathematics'),
(1004, 'James',  'Okonkwo',   'james.okonkwo@lakeside.edu',  2022, 'Computer Science'),
(1005, 'Aisha',  'Patel',     'aisha.patel@lakeside.edu',    2023, 'Data Science'),
(1006, 'Noah',   'Yamamoto',  'noah.yamamoto@lakeside.edu',  2020, 'Mathematics'),
(1007, 'Olivia', 'Kowalski',  'olivia.kowalski@lakeside.edu',2021, 'Computer Science'),
(1008, 'Carlos', 'Reyes',     'carlos.reyes@lakeside.edu',   2022, 'Data Science'),
(1009, 'Mei',    'Lindqvist', 'mei.lindqvist@lakeside.edu',  2023, 'Mathematics'),
(1010, 'Jake',   'OBrien',    'jake.obrien@lakeside.edu',    2020, 'Computer Science');

-- 강좌 데이터 채우기
INSERT INTO edu_dev.silver.courses VALUES
(101, 'Introduction to Programming',   'Computer Science', 3),
(102, 'Data Structures',               'Computer Science', 4),
(103, 'Statistics for Data Science',   'Data Science',     3),
(104, 'Machine Learning Fundamentals', 'Data Science',     4),
(105, 'Calculus I',                    'Mathematics',      4),
(106, 'Linear Algebra',                'Mathematics',      3);

-- 수강 신청 데이터 채우기
INSERT INTO edu_dev.silver.enrollments VALUES
(1,  1001, 101, 'Spring 2023', 8.50),
(2,  1001, 102, 'Fall 2023',   7.80),
(3,  1002, 103, 'Spring 2023', 9.20),
(4,  1002, 104, 'Fall 2023',   6.50),
(5,  1003, 105, 'Spring 2022', 7.00),
(6,  1003, 106, 'Fall 2022',   8.00),
(7,  1004, 101, 'Fall 2022',   6.80),
(8,  1005, 103, 'Spring 2023', 9.00),
(9,  1006, 105, 'Fall 2020',   7.50),
(10, 1007, 102, 'Spring 2022', 8.20),
(11, 1008, 104, 'Fall 2023',   7.30),
(12, 1009, 106, 'Spring 2023', 8.80),
(13, 1010, 101, 'Fall 2021',   5.50),
(14, 1004, 102, 'Spring 2023', 7.10),
(15, 1005, 104, 'Fall 2023',   8.90);

## 연습 3: 뷰 생성

뷰는 테이블 위에 가상 계층을 제공하여 조인된 데이터에 대한 액세스를 단순화하고 비용이 많이 드는 집계를 미리 계산합니다. 이 연습에서는 결합된 수강 신청 레코드를 위한 **표준 뷰**와 미리 집계된 학과 통계를 Lakeside University 분석 팀에 제공하기 위해 `gold` 스키마에 **구체화된 뷰**를 만듭니다.

### 작업 3.1 — 표준 뷰 생성

학생 수강 신청 레코드를 결합하여 사람이 읽기 쉬운 형태로 표현하는 뷰 `edu_dev.silver.vw_student_enrollments`를 만드세요. 다음을 포함해야 합니다:

- 학생 전체 이름(`first_name`과 `last_name`을 공백으로 연결)
- 학생 `email`
- `course_name`
- `department`
- `semester`
- `grade`

> 🤖 **Genie Code 팁:** 물어보세요:
> *"Azure Databricks에서 students, courses, enrollments 세 개의 테이블을 조인하는 SQL 뷰를 작성해 줘"*

**힌트:** 이름을 결합하려면 `CONCAT(first_name, ' ', last_name)` 또는 `||` 연산자를 사용하세요. 세 개의 silver 테이블을 모두 조인해야 합니다.

In [ ]:
%sql
CREATE OR REPLACE VIEW edu_dev.silver.vw_student_enrollments AS
SELECT
  CONCAT(s.first_name, ' ', s.last_name) AS full_name,
  s.email,
  c.course_name,
  c.department,
  e.semester,
  e.grade
FROM edu_dev.silver.enrollments e
JOIN edu_dev.silver.students s ON e.student_id = s.student_id
JOIN edu_dev.silver.courses  c ON e.course_id  = c.course_id;

### 작업 3.2 — 학과 통계를 위한 구체화된 뷰 생성

학과별로 다음을 미리 계산하는 구체화된 뷰 `edu_dev.gold.vw_department_enrollment_stats`를 만드세요:

| 출력 열 | 설명 |
|--------------|-------------|
| `department` | 학과 이름 |
| `total_enrollments` | 총 수강 신청 레코드 수 |
| `avg_grade` | 소수점 둘째 자리로 반올림한 평균 성적 |
| `distinct_students` | 등록된 고유 학생 수 |

> 🤖 **Genie Code 팁:** 물어보세요:
> *"Azure Databricks에서 조인된 두 테이블의 데이터를 집계하는 구체화된 뷰를 만드는 구문은 무엇이야?"*

**힌트:** `CREATE MATERIALIZED VIEW`를 사용하고, `enrollments`를 `courses`와 조인하고, `COUNT()`, `ROUND(AVG(...), 2)`, `COUNT(DISTINCT ...)`를 사용하세요.

> [!NOTE]
> 구체화된 뷰를 실행하려면 SQL 웨어하우스가 필요합니다. 이 셀이 실패하면 노트북 상단의 컴퓨팅 선택기를 확인하고 **Serverless** 컴퓨팅에서 **Serverless SQL 웨어하우스**로 전환하세요.

In [ ]:
%sql
CREATE MATERIALIZED VIEW IF NOT EXISTS edu_dev.gold.vw_department_enrollment_stats AS
SELECT
  c.department,
  COUNT(*)                     AS total_enrollments,
  ROUND(AVG(e.grade), 2)       AS avg_grade,
  COUNT(DISTINCT e.student_id) AS distinct_students
FROM edu_dev.silver.enrollments e
JOIN edu_dev.silver.courses c ON e.course_id = c.course_id
GROUP BY c.department;

**✅ 구체화된 뷰 테스트** — 다음 쿼리를 실행하여 학과별 집계가 반환되는지 확인하세요:

`total_enrollments`, `avg_grade`, `distinct_students` 열과 함께 학과당 한 행이 표시되어야 합니다.

In [ ]:
%sql
SELECT * FROM edu_dev.gold.vw_department_enrollment_stats;

## 연습 4: 볼륨 생성 및 파일 읽기

볼륨은 Unity Catalog 거버넌스를 구조화된 테이블뿐만 아니라 파일로 확장합니다. 이 연습에서는 Lakeside University의 외부 학생 정보 시스템에서 도착하는 파일을 시뮬레이션하여 원시 CSV 파일의 랜딩 영역 역할을 할 **관리형 볼륨**을 `bronze` 스키마에 만듭니다.

### 작업 4.1 — 관리형 볼륨 생성

`edu_dev.bronze` 스키마에 `raw_files`라는 관리형 볼륨을 만드세요. 이는 원시 데이터 파일의 랜딩 영역 역할을 합니다.

> 🤖 **Genie Code 팁:** 물어보세요:
> *"SQL 문을 사용하여 Unity Catalog에서 관리형 볼륨을 만들려면 어떻게 해야 해?"*

**힌트:** `CREATE VOLUME IF NOT EXISTS <catalog>.<schema>.<volume>`을 사용하세요.

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS edu_dev.bronze.raw_files
COMMENT '학생 정보 시스템에서 가져온 원시 CSV 파일의 랜딩 영역';

### 작업 4.2 — 볼륨에 CSV 파일 쓰기

다음 Python 셀은 샘플 학생 CSV 파일을 새 볼륨에 직접 씁니다. **수정 없이 실행하세요.**

In [ ]:
# 샘플 학생 CSV 파일을 볼륨에 쓰기 — 이 셀을 수정 없이 실행하세요
csv_content = """student_id,first_name,last_name,email,enrollment_year,program
1001,Emma,Watson,emma.watson@lakeside.edu,2022,Computer Science
1002,Liam,Ahmed,liam.ahmed@lakeside.edu,2023,Data Science
1003,Sofia,Chen,sofia.chen@lakeside.edu,2021,Mathematics
1004,James,Okonkwo,james.okonkwo@lakeside.edu,2022,Computer Science
1005,Aisha,Patel,aisha.patel@lakeside.edu,2023,Data Science
1006,Noah,Yamamoto,noah.yamamoto@lakeside.edu,2020,Mathematics
1007,Olivia,Kowalski,olivia.kowalski@lakeside.edu,2021,Computer Science
1008,Carlos,Reyes,carlos.reyes@lakeside.edu,2022,Data Science
1009,Mei,Lindqvist,mei.lindqvist@lakeside.edu,2023,Mathematics
1010,Jake,OBrien,jake.obrien@lakeside.edu,2020,Computer Science"""

dbutils.fs.put("/Volumes/edu_dev/bronze/raw_files/students.csv", csv_content, overwrite=True)
print("students.csv를 /Volumes/edu_dev/bronze/raw_files/에 작성했습니다")

### 작업 4.3 — 볼륨에서 CSV를 DataFrame으로 읽기

방금 볼륨에 넣은 CSV 파일을 Spark DataFrame으로 읽어 내용을 표시하세요. 파일은 `/Volumes/edu_dev/bronze/raw_files/students.csv`에 있습니다.

> 🤖 **Genie Code 팁:** 물어보세요:
> *"PySpark에서 Unity Catalog 볼륨 경로의 헤더 행이 있는 CSV 파일을 읽으려면 어떻게 해야 해?"*

**힌트:** `spark.read.csv(path, header=True, inferSchema=True)`를 사용한 다음 결과에 대해 `.display()` 또는 `.show()`를 호출하세요.

In [ ]:
df = spark.read.csv(
    "/Volumes/edu_dev/bronze/raw_files/students.csv",
    header=True,
    inferSchema=True
)
display(df)

## 연습 5: 재사용 가능한 SQL 함수 생성

Unity Catalog를 사용하면 비즈니스 로직을 캡슐화하는 **거버넌스가 적용된 재사용 가능한 함수**를 만들 수 있습니다. 모든 쿼리에서 성적 변환 로직을 반복하는 대신 SQL 함수로 한 번 작성하고 작업 영역의 모든 노트북이나 쿼리에서 호출할 수 있습니다.

### 작업 5.1 — 성적 분류 함수 생성

`grade DECIMAL(4,2)`를 받아 Lakeside University에서 사용하는 척도에 따라 문자 분류를 `STRING`으로 반환하는 SQL 스칼라 함수 `edu_dev.silver.get_grade_classification`을 만드세요:

| 성적 범위 | 분류 |
|-------------|----------------|
| ≥ 8.5 | `'A'` |
| ≥ 7.0 | `'B'` |
| ≥ 5.5 | `'C'` |
| ≥ 4.0 | `'D'` |
| < 4.0 | `'F'` |

> 🤖 **Genie Code 팁:** 물어보세요:
> *"숫자 점수를 기반으로 문자 성적을 반환하기 위해 CASE 식을 사용하는 SQL 스칼라 함수를 Unity Catalog에서 만들려면 어떻게 해야 해?"*

**힌트:** `CREATE FUNCTION ... RETURNS STRING RETURN CASE WHEN ... END`을 사용하세요.

In [ ]:
%sql
CREATE OR REPLACE FUNCTION edu_dev.silver.get_grade_classification(grade DECIMAL(4,2))
RETURNS STRING
COMMENT 'Lakeside University 성적 척도에 따라 문자 성적 분류를 반환합니다'
RETURN CASE
  WHEN grade >= 8.5 THEN 'A'
  WHEN grade >= 7.0 THEN 'B'
  WHEN grade >= 5.5 THEN 'C'
  WHEN grade >= 4.0 THEN 'D'
  ELSE 'F'
END;

### 작업 5.2 — 함수 테스트

`edu_dev.silver.enrollments`를 쿼리하고 새 함수를 적용하여 각 행에 대한 `grade_classification` 열을 생성하세요. `enrollment_id`, `student_id`, `course_id`, `semester`, `grade`, `grade_classification`을 포함하세요.

> 🤖 **Genie Code 팁:** 물어보세요:
> *"Azure Databricks에서 SELECT 문으로 Unity Catalog 사용자 정의 함수를 호출하려면 어떻게 해야 해?"*

In [ ]:
%sql
SELECT
  enrollment_id,
  student_id,
  course_id,
  semester,
  grade,
  edu_dev.silver.get_grade_classification(grade) AS grade_classification
FROM edu_dev.silver.enrollments
ORDER BY enrollment_id;

## 연습 6: DDL 작업

Unity Catalog 객체는 요구 사항이 변경됨에 따라 시간이 지나면서 발전합니다. 이 연습에서는 `ALTER` 문을 사용하여 다음을 수행합니다:
1. 새 열로 `students` 테이블을 확장합니다.
2. `edu_dev` 카탈로그에 거버넌스 **태그**를 적용합니다.

### 작업 6.1 — students 테이블에 열 추가

학생 등록 시스템이 전화번호를 캡처하도록 업데이트되었습니다. `edu_dev.silver.students`에 `STRING` 유형의 null 허용 `phone_number` 열을 추가하세요.

> 🤖 **Genie Code 팁:** 물어보세요:
> *"SQL을 사용하여 Azure Databricks의 기존 Delta 테이블에 새 null 허용 열을 추가하려면 어떻게 해야 해?"*

**힌트:** `ALTER TABLE`을 사용하세요.

In [ ]:
%sql
ALTER TABLE edu_dev.silver.students
ADD COLUMN phone_number STRING
COMMENT '학생 연락처 전화번호';

### 작업 6.2 — 카탈로그에 태그 적용

거버넌스와 검색 가능성을 지원하기 위해 `edu_dev` 카탈로그에 다음 태그를 적용하세요:

| 태그 키 | 태그 값 |
|---------|----------|
| `environment` | `'development'` |
| `university` | `'lakeside'` |
| `data_classification` | `'internal'` |

> 🤖 **Genie Code 팁:** 물어보세요:
> *"Databricks에서 ALTER CATALOG 문을 사용하여 Unity Catalog 카탈로그에 메타데이터 태그를 설정하려면 어떻게 해야 해?"*

In [ ]:
%sql
ALTER CATALOG edu_dev SET TAGS (
  'environment'         = 'development',
  'university'          = 'lakeside',
  'data_classification' = 'internal'
);

### 작업 6.3 — 작업 확인

다음 셀을 실행하여 카탈로그 구조, 테이블 스키마, 태그가 모두 제대로 설정되었는지 확인하세요. 이 랩 전반에 걸쳐 만든 것과 출력을 비교하세요.

In [ ]:
%sql
-- silver 스키마의 모든 테이블과 뷰 표시
SHOW TABLES IN edu_dev.silver;

In [ ]:
%sql
-- students 테이블 설명 — 새 phone_number 열이 있는지 확인
DESCRIBE TABLE edu_dev.silver.students;

In [ ]:
%sql
-- 카탈로그 세부 정보 표시
DESCRIBE CATALOG EXTENDED edu_dev;

In [ ]:
%sql
-- 정보 스키마를 통해 태그 쿼리(구조화된 방법)
SELECT *
FROM system.information_schema.catalog_tags
WHERE catalog_name = 'edu_dev';

---

## 다음 단계

모든 노트북 연습을 완료했습니다. **랩 지침**으로 돌아가 다음을 계속하세요:

- **연습**: AI/BI Genie Space 구성(선택 사항)